[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/whitney_forms/whitney_forms.ipynb)

# Whitney Forms

This example computes Whitney forms that reconstruct continuous fields across triangles and tetrahedra from discrete values on vertices, edges, and faces, and applies them to steady electric conduction in an anisotropic sheet. A potential stored on the vertices spreads linearly across each triangle so adjacent triangles meet continuously along shared edges. A circulation stored on the edges spreads into a directional field whose tangential reading along each edge agrees from both neighbouring triangles even where the transverse component jumps. On a tetrahedron, the same construction adds face forms that carry prescribed flux straight out from the opposite corner, while the volume pairing of edge forms with their curls measures topological linkage between opposite edges. In steady conduction, the material law acts on the reconstructed field inside each triangle and returns vertex and edge currents across the triangulation.

A position is modelled as an intrinsic point, so joining two boundary edges gives a triangle's signed area patch `(sides[0] & sides[1]) / 2`. The term ***simplex gradient*** here means the extrinsic plane `sides / (2 * patches)` obtained by dividing each opposite edge by twice that patch, which pairs with displacements via `gradients & displacement` without a metric, while dividing the offset from the opposite corner `(points - corners) / (2 * patches)` builds the edge 1-form the same way. In 3D projective space, joining the corners off a sub-simplex and dividing by all four corners joined `complements / (complements & spans)` yields the Whitney forms of every degree in one quotient. Assembled across the triangulation, these reconstruction maps commute with the coboundary operators (`whitney1(d0(potential)) == gradient(potential)`), and the material conductivity enters only through `gradient.adjugate()((areas * conductivity)(gradient))` to pull local currents back to vertex conductances.

In finite element exterior calculus notation, a simplex is parameterized by barycentric coordinates $\lambda_i(x)$ satisfying $\sum_i \lambda_i = 1$, and the Whitney $k$-form of an oriented $k$-simplex $[i_0, \dots, i_k]$ is written as the alternating sum $w_{i_0 \dots i_k} = k! \sum_{m=0}^k (-1)^m \lambda_{i_m}\, d\lambda_{i_0} \wedge \dots \wedge d\lambda_{i_{m-1}} \wedge d\lambda_{i_{m+1}} \wedge \dots \wedge d\lambda_{i_k}$, spanning the trimmed spaces $\mathcal{P}_1^- \Lambda^k$ with $d W^k = W^{k+1} d$. Assembling the anisotropic conductance matrix then requires reference-element Jacobians and coordinate inner products $K_{pq} = \int_\Omega (\nabla \lambda_p)^T \sigma (\nabla \lambda_q)\, dA$ on each triangle.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from functools import reduce
from operator import and_

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA2D as ga, PGA3D as space
from numga.extensor import Extensor
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology, simplex_parity
from examples.surfaces.whitney_forms import render, scenarios
from examples.util import shapes
from examples.util.animation import save_figure

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Vector+
Point = ga.gatype.antivector()                                               # Antivector-
Patch = ga.gatype.antibivector()                                             # Antibivector-
Conductivity = ga.gatype((Point, Plane))                                     # Point- <- Plane+

# The tetrahedron's projective space gives corners unit weight so their joins span planes, lines, and points.
space_mv = NumpyContext(space).multivector
SpaceScalar = space.gatype.scalar()                                          # Scalar+
SpacePoint = space.gatype.antivector()                                       # Antivector-

np.set_printoptions(precision=4, suppress=True)

## Simplex gradients and vertex forms

Each vertex 0-form rises linearly from zero along the opposite edge to one at its own corner, so adjacent triangles meet continuously along their shared edge even as their slopes jump across the seam. Pairing a corner's simplex gradient with a displacement via `gradients & displacement` measures the linear change directly from the triangle's boundary edges and signed area patch.

In [ ]:
n_triangle_subdivisions = 8
n_star_divisions = 2
disk_radius = 1.0

In [ ]:
def simplex_gradients(topology: Topology, vertices: Point) -> tuple[Patch, Plane]:
    """Each triangle's boundary edges determine its signed area patch (`Patch-[F]`) and three corner simplex gradients (`[3] Plane+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    # Dividing each opposite edge by twice the signed area patch gives the simplex gradient
    # that rises by one from that edge to its corner without using a metric.
    gradients = sides / (2 * patches)                                        # [3] Plane+[F]
    return patches, gradients


def vertex_whitney_forms(topology: Topology, vertices: Point, points: Point) -> Scalar:
    """Each corner's simplex gradient evaluates the affine vertex 0-forms (`[3, ...] Scalar+[F]`) across `points`."""
    corners = topology.incidence[2, 0] * vertices                            # [3] Point-[F]
    patches, gradients = simplex_gradients(topology, vertices)               # Patch-[F], [3] Plane+[F]
    return 1 + (gradients[:, None] & (points[None, :] - corners[:, None]))   # [3, ...] Scalar+[F]


triangle, triangle_vertices, sample_points = scenarios.single_triangle(n_triangle_subdivisions)
patches, gradients = simplex_gradients(triangle, triangle_vertices)          # Patch-[F], [3] Plane+[F]
vertex_forms = vertex_whitney_forms(triangle, triangle_vertices, sample_points)  # [3, ...] Scalar+[F]

star_topology, star_coordinates = shapes.hexagonal_disk(disk_radius, n_star_divisions)
V_star, E_star, F_star = star_topology.n_elements
star_vertices = mv.vector(star_coordinates).dual().field()                   # Point-[V_star]
vertex_form = mv.scalar((np.arange(V_star) == 0)[..., None]).field()         # Scalar+[V_star]

save_figure(
    render.vertex_forms_figure(triangle, triangle_vertices, gradients, star_topology, star_vertices, vertex_form),
    "whitney_vertex_0form",
);

The orange arrows point along each corner's constant simplex gradient, while the blue shading and level lines trace the vertex 0-form's height from zero on the bold opposite edge to one at the highlighted corner. Across the six-triangle vertex star on the right, the local linear pieces join into a continuous pyramid around the central vertex.

In finite element notation, the three vertex 0-forms are the barycentric hat functions $\lambda_i$, with piecewise-constant differentials $d\lambda_i$ on each triangle.

In [ ]:
print("Triangle area:", patches.antinorm().sites.sum().to_array())

## Edge forms

An edge 1-form measures line integrals along displacements. Dividing the radial offset from the opposite corner `(points - corners) / (2 * patches)` by twice the signed patch gives a linear 1-form that pairs to zero along both edges meeting at that corner and integrates to one along the opposite edge. Because its tangential reading depends only on the shared edge vector, two triangles meeting along an interior edge agree tangentially while their normal components remain free to jump.

In [ ]:
def edge_whitney_forms(topology: Topology, vertices: Point, points: Point) -> Plane:
    """Radial displacements from each opposite corner over twice the signed patch define the three local edge Whitney 1-forms (`[3, ...] Plane+[F]`) at `points`."""
    corners = topology.incidence[2, 0] * vertices                            # [3] Point-[F]
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    # Along the two edges meeting at a corner, the radial displacement from that corner is parallel
    # to the edge, while along the opposite edge its join with the side equals twice the patch.
    return (points[None, :] - corners[:, None]) / (2 * patches)              # [3, ...] Plane+[F]


def edge_whitney_map(topology: Topology, vertices: Point, points: Point) -> SparseExtensor:
    """Local edge 1-forms weighted by oriented face boundaries assemble the evaluation map (`[...] (Plane+ <- Scalar+)[F <- E]`) at `points`."""
    forms = edge_whitney_forms(topology, vertices, points)                   # [3, ...] Plane+[F]
    boundaries = topology.boundary[2, 1][:, None]                            # [3, 1] Scalar+[F <- E]
    return (spdiag(forms) * boundaries).sum(axis=0) * Scalar                 # [...] (Plane+ <- Scalar+)[F <- E]


triangle_corners = triangle.incidence[2, 0] * triangle_vertices              # [3] Point-[F]
edge_forms = edge_whitney_forms(triangle, triangle_vertices, sample_points)  # [3, ...] Plane+[F]
corner_forms = edge_whitney_forms(triangle, triangle_vertices, triangle_corners)  # [3, 3] Plane+[F]

diamond_topology, diamond_vertices = scenarios.edge_diamond()
V_diamond, E_diamond, F_diamond = diamond_topology.n_elements
diamond_corners = diamond_topology.incidence[2, 0] * diamond_vertices        # [3] Point-[F_diamond]
shared_circulation = mv.scalar(np.isin(np.arange(E_diamond), diamond_topology.interior[1])[..., None]).field()  # Scalar+[E_diamond]
shared_corner_forms = edge_whitney_map(diamond_topology, diamond_vertices, diamond_corners)(shared_circulation)  # [3] Plane+[F_diamond]

diamond_edges = diamond_topology.d[0] * diamond_vertices                     # Point-[E_diamond]
edge_samples = ((diamond_topology.incidence[1, 0] * np.array([[0.15], [0.85]])).sum(axis=0)) * diamond_vertices  # Point-[E_diamond]
shared_reading = diamond_topology.reading(diamond_topology.interior[1], 1)   # Scalar+[1 <- E_diamond]
face_from_shared = (diamond_topology.incidence[2, 1] * shared_reading.adjugate()).sum(axis=0)  # Scalar+[F_diamond <- 1]
seam_points = face_from_shared * (shared_reading * edge_samples)             # Point-[F_diamond]
seam_edges = face_from_shared * (shared_reading * diamond_edges)             # Point-[F_diamond]
seam_readings = edge_whitney_map(diamond_topology, diamond_vertices, seam_points[None])[0](shared_circulation)  # Plane+[F_diamond]

save_figure(
    render.edge_forms_figure(triangle, triangle_vertices, corner_forms, diamond_topology, diamond_vertices, shared_corner_forms),
    "whitney_edge_1form",
);

The highlighted orange edge carries unit circulation, and the feathered streaks trace the metric-dual directions of each edge 1-form with length proportional to magnitude. Across the two-triangle diamond on the right, the streaks bend across the vertical seam while maintaining matching tangential components along the shared edge.

In the differential-form notation of finite element exterior calculus, the edge 1-form is $w_{ij} = \lambda_i\, d\lambda_j - \lambda_j\, d\lambda_i$, with exterior derivative $dw_{ij} = 2\, d\lambda_i \wedge d\lambda_j$.

In [ ]:
print("Normal reading (left, right):", (seam_readings & mv.x.dual()).to_array())

## Tetrahedral forms and edge linkage

On a tetrahedron in 3D projective space, points carry unit weight so joining corners spans the edges, faces, and volume between them. Joining the corners off a sub-simplex and dividing by the join with its own corners `complements / (complements & spans)` yields the Whitney forms for vertices, edges, and faces from one quotient. Evaluating `forms & point` produces the vertex 0-form height, the plane spanned by the opposite edge and `point` for an edge 1-form, and the line radiating from the opposite corner through `point` for a face 2-form. Wedging each edge 1-form at the centroid with twice another edge 1-form and pairing with the tetrahedral volume gives a topological helicity matrix that links only opposite edges.

In [ ]:
n_tet_subdivisions = 6
n_pages = 5
n_face_cuts = 4

In [ ]:
def tetrahedral_forms(topology: Topology, vertices: SpacePoint, degree: int) -> Extensor:
    """Dividing the join of the corners off each `degree`-simplex by all four corners joined gives the tetrahedral Whitney forms (`[...] Extensor+[T]`)."""
    # Reordering the off-simplex corners flips numerator and denominator alike and cancels.
    complements = reduce(and_, topology.off[3, degree] * vertices)
    spans = reduce(and_, topology.on[3, degree] * vertices)
    return complements / (complements & spans)


def tetrahedral_helicity(topology: Topology, vertices: SpacePoint) -> SpaceScalar:
    """Each edge's Whitney 1-form paired at the centroid with each edge's constant curl integrates to the symmetric volume helicity (`[6, 6] Scalar+[T]`)."""
    corners = topology.incidence[3, 0] * vertices                            # [4] Point-[T]
    forms = tetrahedral_forms(topology, vertices, 1)                         # [6] Bivector+[T]
    # Each edge 1-form is linear across the tetrahedron while its curl equals twice the form and stays constant,
    # so evaluating the 1-form at the centroid integrates their wedge product over the tetrahedron exactly.
    curls = 2 * forms                                                        # [6] Bivector+[T]
    centroid_forms = forms & corners.mean(axis=0)                            # [6] Plane+[T]
    densities = ((centroid_forms[:, None] ^ curls[None, :]) + (centroid_forms[None, :] ^ curls[:, None])) / 2  # [6, 6] Trivector+[T]
    volume = reduce(and_, corners[1:] - corners[0]) / 6                      # Volume-[T]
    return densities & volume                                                # [6, 6] Scalar+[T]


tetrahedron, tet_vertices, tet_points = scenarios.single_tetrahedron(n_tet_subdivisions)
tet_vertex_forms, tet_edge_forms, tet_face_forms = (
    tetrahedral_forms(tetrahedron, tet_vertices, degree) for degree in range(3)
)
tet_helicity = tetrahedral_helicity(tetrahedron, tet_vertices)               # [6, 6] Scalar+[T]

In [ ]:
def tetrahedral_edge_pages(topology: Topology, vertices: SpacePoint, n_pages: int) -> SpacePoint:
    """Evenly spaced points along each edge span `n_pages` level-surface triangles (`[6, n_pages, 3] Point-[T]`) with the opposite edge."""
    corners = topology.incidence[3, 0] * vertices                            # [4] Point-[T]
    tails, heads = topology.corners[3, 1]
    opposite_tails, opposite_heads = topology.complements[3, 1]
    fractions = np.arange(1, n_pages + 1) / (n_pages + 1)                    # [n_pages]
    points = corners[tails, None] * (1 - fractions) + corners[heads, None] * fractions  # [6, n_pages] Point-[T]
    # At any point, the edge 1-form evaluates to the plane through that point and the opposite edge,
    # so every triangle spanned by the opposite edge and an edge point is a level surface of the form.
    opposite_tail = corners[opposite_tails, None].broadcast_to(points.shape)  # [6, n_pages] Point-[T]
    opposite_head = corners[opposite_heads, None].broadcast_to(points.shape)  # [6, n_pages] Point-[T]
    return stack((opposite_tail, points, opposite_head), axis=-1)            # [6, n_pages, 3] Point-[T]


def tetrahedral_face_lines(topology: Topology, vertices: SpacePoint, face_weights: np.ndarray) -> SpacePoint:
    """Straight segments from each tetrahedral corner to equal-area subtriangle centres on the opposite face trace the face 2-form field lines (`[4, ..., 2] Point-[T]`)."""
    corners = topology.incidence[3, 0] * vertices                            # [4] Point-[T]
    facets = corners[topology.complements[3, 0].T]                           # [4, 3] Point-[T]
    crossings = (facets[:, None] * face_weights).sum(axis=-1)                # [4, ...] Point-[T]
    # Evaluating a face 2-form at any point yields the line through the opposite corner and that point,
    # so straight segments from the opposite corner carry equal shares of the uniform face flux.
    return stack((corners[:, None].broadcast_to(crossings.shape), crossings), axis=-1)  # [4, ..., 2] Point-[T]


tet_pages = tetrahedral_edge_pages(tetrahedron, tet_vertices, n_pages)       # [6, n_pages, 3] Point-[T]
tet_lines = tetrahedral_face_lines(tetrahedron, tet_vertices, scenarios.face_part_middles(n_face_cuts))  # [4, ..., 2] Point-[T]

save_figure(
    render.tetrahedron_forms_figure(tetrahedron, tet_vertices, tet_pages, tet_lines),
    "whitney_tet_forms",
);

In the left panel, translucent triangular pages fan around the blue opposite edge to mark evenly spaced circulation levels of the orange edge's 1-form. In the middle panel, straight field lines radiate from the highlighted opposite corner through equal-area subtriangles of the tinted face, each carrying an equal share of the face 2-form's unit flux. In the right panel, coloured loops around the blue edge are threaded by the orange edge, illustrating the topological linkage captured by the volume helicity pairing.

In Whitney-form notation, the edge and face forms are written $w_{ij} = \lambda_i\, d\lambda_j - \lambda_j\, d\lambda_i$ and $w_{ijk} = 2(\lambda_i\, d\lambda_j \wedge d\lambda_k + \lambda_j\, d\lambda_k \wedge d\lambda_i + \lambda_k\, d\lambda_i \wedge d\lambda_j)$, and their mutual helicity $\frac{1}{2}\int_T (w_e \wedge dw_f + w_f \wedge dw_e)$ evaluates to $\pm\frac{1}{6}$ on opposite edges $e, f$ and zero on adjacent edges.

## Material response and anisotropic conduction

Assembled across a triangulated domain, the local reconstruction maps turn vertex potentials and edge circulations into face fields that commute with the topological coboundary operators (`whitney1(d0(potential)) == gradient(potential)` and `curl(cochain) == whitney2(d1(cochain))`). A material conductivity maps potential gradients to current directions on each triangle, and pulling it back through `gradient.adjugate()` produces the symmetric vertex conductance without altering the topological operators.

In [ ]:
n_disk_divisions = 9
electrode_threshold = 0.8
fibre_angle = np.pi / 6
principal_conductivities = np.array([[1, 1], [10, 1]])                       # [2, 2]

In [ ]:
def simplex_gradient_map(topology: Topology, vertices: Point) -> SparseExtensor:
    """Corner simplex gradients summed by face incidence map vertex potentials to piecewise-constant face gradients (`(Plane+ <- Scalar+)[F <- V]`)."""
    patches, gradients = simplex_gradients(topology, vertices)               # Patch-[F], [3] Plane+[F]
    return (spdiag(gradients) * topology.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[F <- V]


def cochain_maps(topology: Topology, vertices: Point) -> tuple[SparseExtensor, SparseExtensor, SparseExtensor]:
    """Corner simplex gradients and signed patches assemble the centroid edge 1-form map (`(Plane+ <- Scalar+)[F <- E]`), exterior curl (`(Bivector+ <- Scalar+)[F <- E]`), and face 2-form map (`(Bivector+ <- Scalar+)[F]`)."""
    patches, gradients = simplex_gradients(topology, vertices)               # Patch-[F], [3] Plane+[F]
    tails, heads = topology.corners[2, 1]
    # At each face centroid all three vertex 0-forms equal 1/3, reducing each edge 1-form
    # to one third of the difference between its endpoint simplex gradients.
    centroid_forms = (gradients[heads] - gradients[tails]) / 3               # [3] Plane+[F]
    whitney1 = (spdiag(centroid_forms) * topology.oriented[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[F <- E]
    curls = 2 * (gradients[tails] ^ gradients[heads])                        # [3] Bivector+[F]
    curl = (spdiag(curls) * topology.oriented[2, 1]).sum(axis=0) * Scalar    # (Bivector+ <- Scalar+)[F <- E]
    whitney2 = mv.xy / (mv.xy & patches) * Scalar                            # (Bivector+ <- Scalar+)[F]
    return whitney1, curl, whitney2


topology, coordinates = shapes.hexagonal_disk(disk_radius, n_disk_divisions)
V, E, F = topology.n_elements
vertices = mv.vector(coordinates).dual().field()                             # Point-[V]
d0 = topology.d[0] * Scalar                                                  # (Scalar+ <- Scalar+)[E <- V]
d1 = topology.d[1] * Scalar                                                  # (Scalar+ <- Scalar+)[F <- E]
gradient = simplex_gradient_map(topology, vertices)                          # (Plane+ <- Scalar+)[F <- V]
whitney1, curl, whitney2 = cochain_maps(topology, vertices)

In [ ]:
def steady_conduction(
    topology: Topology,
    vertices: Point,
    conductivity: Conductivity,
    electrodes: np.ndarray,
    prescribed: Scalar,
) -> tuple[Scalar, Point]:
    """Pulling area-weighted conductivity back through the simplex gradient determines the steady vertex potential (`[...] Scalar+[V]`) and face current (`[...] Point-[F]`) for `prescribed` electrode potentials."""
    V, E, F = topology.n_elements
    patches, corner_gradients = simplex_gradients(topology, vertices)        # Patch-[F], [3] Plane+[F]
    areas = patches.antinorm()                                               # Scalar+[F]
    gradient = (spdiag(corner_gradients) * topology.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[F <- V]
    # Pulling the area-weighted conductivity back through the simplex gradient gives the vertex conductance.
    medium = areas * conductivity                                            # [...] (Point- <- Plane+)[F]
    conductance = gradient.adjugate()(medium(gradient))                      # [...] (Antiscalar- <- Scalar+)[V <- V]
    free_vertices = np.setdiff1d(np.arange(V), electrodes)
    free = topology.reading(free_vertices, 0).adjugate() * Scalar            # (Scalar+ <- Scalar+)[V <- V_free]
    rhs = -free.adjugate()(conductance(prescribed))                          # [...] Antiscalar-[V_free]
    potential = prescribed + free(free.adjugate()(conductance(free)).solve(rhs))  # [...] Scalar+[V]
    current = -conductivity(gradient(potential))                             # [...] Point-[F]
    return potential, current


fibre_turn = (mv.xy * (-fibre_angle / 2)).exp()                              # [] Rotor+
fibre = fibre_turn >> mv.x.dual()                                            # [] Point-
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                                  # [] Rotor+
directions = stack((fibre, quarter_turn >> fibre))                           # [2] Point-
conductivity = (directions * (Plane & directions) * principal_conductivities).sum(axis=-1)  # [2] Point- <- Plane+

boundary = topology.exterior[0]
exterior = topology.reading(boundary, 0)                                     # Scalar+[V_exterior <- V]
electrodes = boundary[(exterior * (mv.x & vertices)).abs() > electrode_threshold]
pinned = topology.reading(electrodes, 0)                                     # Scalar+[V_pinned <- V]
prescribed = pinned.adjugate() * (pinned * (mv.x & vertices))                # Scalar+[V]
potential, current = steady_conduction(topology, vertices, conductivity, electrodes, prescribed)  # [2] Scalar+[V], [2] Point-[F]

save_figure(
    render.conduction_figure(topology, vertices, potential, current, fibre),
    "whitney_anisotropic_flow",
);

The left disk conducts isotropically from the left electrode to the right electrode, while the right disk conducts ten times more strongly along the dashed fibre chords and bends the current streaks toward the fibre axis. Background shading and contours show the electric potential on a shared colour scale, and the feathered streaks share a common speed scale across both disks.

In finite element matrix notation, pulling the conductivity tensor back through the element gradients assembles the stiffness matrix $K_{pq} = \int_\Omega (\nabla \lambda_p)^T \sigma (\nabla \lambda_q)\, dA$, which reduces for isotropic conductivity to the cotangent Laplacian.

In [ ]:
first_face = topology.reading(np.array([0]), 2)                              # Scalar+[1 <- F]
print("Centre current (isotropic vs anisotropic):\n", (stack((mv.x, mv.y)) & (first_face * current)[:, None]).to_array())

Evaluating the linear edge 1-forms at the three edge midpoints integrates quadratic products across each triangle exactly, keeping the 1-form conductance positive-definite even on obtuse triangles where circumcentric discrete exterior calculus assigns a negative edge weight.

In [ ]:
def edge_conductance(topology: Topology, vertices: Point, conductivity: Conductivity) -> SparseExtensor:
    """Three-midpoint quadrature integrates quadratic products of linear edge 1-forms exactly into the edge conductance (`[...] (Antiscalar- <- Scalar+)[E <- E]`)."""
    patches, corner_gradients = simplex_gradients(topology, vertices)        # Patch-[F], [3] Plane+[F]
    areas = patches.antinorm()                                               # Scalar+[F]
    midpoints = topology.incidence[2, 1] * (topology.incidence[1, 0].mean(axis=0) * vertices)  # [3] Point-[F]
    midpoint_map = edge_whitney_map(topology, vertices, midpoints)           # [3] (Plane+ <- Scalar+)[F <- E]
    weights = ((areas / 3) * conductivity)[..., None]                        # [..., 3] (Point- <- Plane+)[F]
    return midpoint_map.adjugate()(weights(midpoint_map)).sum(axis=-1)       # [...] (Antiscalar- <- Scalar+)[E <- E]


edge_conductances = edge_conductance(topology, vertices, conductivity)       # [2] (Antiscalar- <- Scalar+)[E <- E]

obtuse_topology, obtuse_vertices = scenarios.obtuse_diamond()
V_obtuse, E_obtuse, F_obtuse = obtuse_topology.n_elements
obtuse_conductance = edge_conductance(obtuse_topology, obtuse_vertices, Plane.dual())  # (Antiscalar- <- Scalar+)[E_obtuse <- E_obtuse]
shared_cochain = mv.scalar(np.isin(np.arange(E_obtuse), obtuse_topology.interior[1])[..., None]).field()  # Scalar+[E_obtuse]
obtuse_energy = (shared_cochain & obtuse_conductance(shared_cochain)).sites.sum()  # [] Scalar+

In [ ]:
# checks
corner_values = vertex_whitney_forms(triangle, triangle_vertices, triangle_corners)
assert ((corner_values - np.eye(3)).abs() < 1e-10).all()
assert ((vertex_forms.sum(axis=0) - 1).abs() < 1e-10).all()
assert (gradients.sum(axis=0).norm() < 1e-10).all()
assert ((vertex_whitney_forms(triangle, triangle_vertices, triangle_corners.mean(axis=0)[None]) - 1 / 3).abs() < 1e-10).all()

tails, heads = triangle.corners[2, 1]
alternating_forms = triangle.orientation[1][0, :, None] * (
    vertex_forms[tails] * gradients[heads, None] - vertex_forms[heads] * gradients[tails, None]
)
assert ((edge_forms - alternating_forms).norm() < 1e-10).all()
sides = triangle.boundary[2, 1] * (triangle.d[0] * triangle_vertices)
triangle_midpoints = triangle.incidence[2, 1] * (triangle.incidence[1, 0].mean(axis=0) * triangle_vertices)
midpoint_readings = edge_whitney_forms(triangle, triangle_vertices, triangle_midpoints) & sides[None, :]
assert ((midpoint_readings - np.eye(3)).abs() < 1e-10).all()
assert (((seam_readings & seam_edges) - 1).abs() < 1e-10).all()

for degree, forms in enumerate((tet_vertex_forms, tet_edge_forms, tet_face_forms)):
    spans = reduce(and_, tetrahedron.on[3, degree] * tet_vertices)
    assert (((forms[:, None] & spans[None, :]) - np.eye(spans.shape[0])).abs() < 1e-10).all()
tet_weights = tet_vertex_forms[:, None] & tet_points[None, :]
assert ((tet_weights.sum(axis=0) - 1).abs() < 1e-10).all()

tet_corners = tetrahedron.incidence[3, 0] * tet_vertices
tet_tails, tet_heads = tetrahedron.corners[3, 1]
alternating_tet_edges = tet_weights[tet_tails] * tet_vertex_forms[tet_heads, None] - tet_weights[tet_heads] * tet_vertex_forms[tet_tails, None]
edge_spans = reduce(and_, tetrahedron.on[3, 1] * tet_vertices)
assert ((((tet_edge_forms[:, None] & tet_points[None, :]) - alternating_tet_edges)[:, :, None] & tet_corners[None, None, :]).abs() < 1e-10).all()
assert (((tet_edge_forms - (tet_vertex_forms[tet_tails] ^ tet_vertex_forms[tet_heads]))[:, None] & edge_spans[None, :]).abs() < 1e-10).all()
face_first, face_second, face_third = tetrahedron.corners[3, 2]
face_spans = reduce(and_, tetrahedron.on[3, 2] * tet_vertices)
assert (((tet_face_forms - (tet_vertex_forms[face_first] ^ tet_vertex_forms[face_second] ^ tet_vertex_forms[face_third]))[:, None] & face_spans[None, :]).abs() < 1e-10).all()

deformed_helicity = tetrahedral_helicity(tetrahedron, scenarios.deformed_tetrahedron())
assert ((tet_helicity - deformed_helicity).abs() < 1e-10).all()
edge_corners = tetrahedron.corners[3, 1].T
edge_pairs = np.concatenate(np.broadcast_arrays(edge_corners[:, None], edge_corners[None, :]), axis=-1)
disjoint = (edge_corners[:, None, :, None] != edge_corners[None, :, None, :]).all(axis=(-1, -2))
linking = np.where(disjoint, simplex_parity(edge_pairs), 0) / 6
assert ((tet_helicity - linking).abs() < 1e-10).all()

linear_potential = (mv.x & vertices) * 0.7 - (mv.y & vertices) * 1.1
assert (d1(d0(linear_potential)).abs() < 1e-10).all()
assert ((whitney1(d0(linear_potential)) - gradient(linear_potential)).norm() < 1e-10).all()
trial_cochain = d0(linear_potential) + mv.scalar(np.sin(np.arange(E))[..., None]).field()
assert ((curl(trial_cochain) - whitney2(d1(trial_cochain))).norm() < 1e-10).all()

disk_patches, disk_gradients = simplex_gradients(topology, vertices)
conductance = gradient.adjugate()((disk_patches.antinorm() * conductivity)(gradient))
quadratic_potential = (mv.x & vertices).squared() - 0.5 * (mv.y & vertices)
assert ((d0.adjugate()(edge_conductances(d0(quadratic_potential))) - conductance(quadratic_potential)).antinorm() < 1e-10).all()
assert (((linear_potential & conductance(quadratic_potential)) - (quadratic_potential & conductance(linear_potential))).sites.sum().abs() < 1e-10).all()
assert (obtuse_energy > 0).all()

free_vertices = np.setdiff1d(np.arange(V), electrodes)
free = topology.reading(free_vertices, 0).adjugate() * Scalar
assert (free.adjugate()(conductance(potential)).antinorm() < 1e-9).all()